In [ ]:
# Only in Colab: work inside the homework folder of Drive (plotter.py is there)
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir("/content/drive/MyDrive/POSGRADOS/HUNGRIA/MASTER/1/FNANCIAL PROCESS /HOMEWORKS/2_HW_LECTURE_4")
except ImportError:
    pass
print(os.getcwd())

**Homework 2: Implement calibrator for Tree. Part 1: Background**

*Background: In the Black--Scholes model volatility is a key parameter that determines option price. In a Balanced Binomial Tree (i.e. risk-neutral up step probability $= 0.5$, risk-neutral down step probability $= 0.5$) the same role is played by step size: higher step is like higher volatility, it results in higher option prices. Also, there is a 1-1 correspondence between step size and price if all other parameters are fixed.*

Claims: **(C1)** the step size plays the role of the volatility; **(C2)** a higher step gives a higher call price; **(C3)** step size and price determine each other (1-1).

**Professor's code** (`Binomial_tree_coding.ipynb`, cells 1, 3, 5, 8 and 13), unchanged

In [ ]:
import numpy as np
from scipy.optimize import minimize
from collections.abc import Callable


def european_call_payoff(S: float, K: float) -> float:
    return max(S-K, 0.0)


def create_spot_tree(spot: float, spot_mult_up: float, spot_mult_down: float, steps: int) -> list[list[float]]:
    previous_level = [spot]
    tree = [previous_level]
    for _ in range(steps):
        new_level = [s * spot_mult_down for s in previous_level]
        new_level += [previous_level[-1] * spot_mult_up]
        tree += [new_level]
        previous_level = new_level
    return tree


def create_discounted_price_tree(spot_tree: list[list[float]], discount_factor: float, K: float, diag: int = 0) -> list[list[float]]:
    spot = spot_tree[0][0]
    spot_mult_up = spot_tree[1][-1]
    spot_mult_down = spot_tree[1][0]
    p_up = ((1 / discount_factor - spot_mult_down) /
                   (spot_mult_up - spot_mult_down))
    p_down = 1 - p_up
    steps = len(spot_tree) - 1
    continuation_value_tree = [[np.nan for _ in level] for level in spot_tree]
    if diag > 0:
        print("risk-neutral measure: ")
        print(('%.3f' % p_up, '%.3f' % p_down))
        # init delta tree
        delta_tree = [[np.nan for _ in level] for level in spot_tree[:-1]] #delta makes no sense for leaves
    # going backwards, payoff is known in leaves
    for i in range(len(spot_tree[-1])):
        spot = spot_tree[-1][i]
        discounted_continuation_value = discount_factor**(steps) * european_call_payoff(spot, K)
        continuation_value_tree[-1][i] = discounted_continuation_value
    for step in range(steps - 1, -1, -1):
        for i in range(len(spot_tree[step])):
            continuation_value_tree[step][i] = p_up * continuation_value_tree[step + 1][i] + \
                                            p_down * continuation_value_tree[step + 1][i + 1]
            if diag > 0:
                delta_tree[step][i] = ((continuation_value_tree[step + 1][i] - continuation_value_tree[step + 1][i + 1]) 
                                       / (spot_tree[step + 1][i] - spot_tree[step + 1][i + 1]))
    if diag > 0:
        print("delta: ")
        delta_tree_readable = [['%.3f' % e for e in n] for n in delta_tree]
        print(delta_tree_readable)
    return continuation_value_tree


def calcBalancedDownStep(spot_mult_up: float, discount_factor: float) -> (float, float):
    return spot_mult_up - 2 * (spot_mult_up - 1 / discount_factor)

**Test of the professor's code** (cells 6 and 14): balanced tree, $p_{up} = p_{down} = 0.5$

In [ ]:
# test of the lecture code (cells 6 and 14): balanced 2-step tree, u = 1.2, DF = 0.95, K = 1
spot, spot_mult_up, steps, discount_factor, strike = 1, 1.2, 2, 0.95, 1
spot_mult_down_balanced = calcBalancedDownStep(spot_mult_up, discount_factor)
print("spot_mult_down: " + str('%.3f' % spot_mult_down_balanced))
spot_tree = create_spot_tree(spot, spot_mult_up, spot_mult_down_balanced, steps)
price_tree = create_discounted_price_tree(spot_tree, discount_factor, strike, 1)
print("price tree: " + str([['%.3f' % e for e in n] for n in price_tree]))

**Homework 1** (HW 2/2, NEW VERSION): the Black-Scholes call pricer, used for the comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from typing import Dict, List, Union

"""The pricing function of European call option"""
# change # 2
#Delete.    def black_scholes_eur_call(r: float, T: float, S0: float, sigma: float,
#Delete.                               K: Union[float, List[float], np.ndarray]) -> np.ndarray:
# replace by
def black_scholes_eur_call(r: float, T: float, S0: float, sigma: float,
                           K: Union[float, List[float], np.ndarray],
                           return_greeks: bool = False):
# end change # 2
    """
    Black-Scholes pricer of European call option on non-dividend-paying stock

    param r: risk-free interest rate (which is constant)
    param T: time to maturity (in years)
    param S0: initial spot price of the underlying stock
    param sigma: volatility of the underlying stock
    param K: strike price (or prices)
    param return_greeks: if True, return a dict with the price and the greeks
    """
    # check conditions
    assert sigma > 0

    K = np.array([K]) if isinstance(K, float) else np.array(K)

    d1_vec = ( np.log( S0 / K ) + ( r + 0.5 * sigma**2 ) * T ) / ( sigma * T**0.5 )
    d2_vec = d1_vec - sigma * T**0.5

    N_d1_vec = norm.cdf(d1_vec)
    N_d2_vec = norm.cdf(d2_vec)

    # change # 1
    #Delete.    return N_d1_vec * S0 - K_vec * np.exp((-1.0)*r*T) * N_d2_vec
    # replace by
    price = N_d1_vec * S0 - K * np.exp((-1.0)*r*T) * N_d2_vec
    # end change # 1

    # change # 2
    if not return_greeks:
        return price
    n_d1_vec = norm.pdf(d1_vec)                  # N'(d1)
    return {
        "price": price,
        "delta": N_d1_vec,
        "gamma": n_d1_vec / (S0 * sigma * T**0.5),
        "vega":  S0 * n_d1_vec * T**0.5,
        "theta": -S0 * n_d1_vec * sigma / (2 * T**0.5) - r * K * np.exp(-r*T) * N_d2_vec,
        "rho":   K * T * np.exp(-r*T) * N_d2_vec,
    }
    # end change # 2

**NEW CODE**: price of a call in a balanced tree (with the lecture code), and the up step for a given tree volatility $\sigma_{tree} = (u - 1/DF)/\sqrt{\Delta t}$

In [ ]:
def balanced_tree_call_price(spot: float, strike: float, maturity: float, steps: int,
                             df_annual: float, spot_mult_up: float) -> float:
    """Price of a European call in a Balanced Binomial Tree (p_up = p_down = 0.5)."""
    discount_factor = df_annual ** (maturity / steps)     # per step: keeps the rate flat
    spot_mult_down = calcBalancedDownStep(spot_mult_up, discount_factor)
    spot_tree = create_spot_tree(spot, spot_mult_up, spot_mult_down, steps)
    price_tree = create_discounted_price_tree(spot_tree, discount_factor, strike)
    return price_tree[0][0]


def up_step_from_tree_vol(sigma_tree: float, maturity: float, steps: int,
                          df_annual: float) -> float:
    """Up step u with step size u - 1/DF = sigma_tree * sqrt(dt)."""
    dt = maturity / steps
    discount_factor = df_annual ** dt
    return 1 / discount_factor + sigma_tree * np.sqrt(dt)

**(C1)** One step of a balanced tree: mean $1/DF$ and standard deviation $u - 1/DF$

In [ ]:
# (C1) one step of a balanced tree: mean and standard deviation of the factor S_{n+1}/S_n
discount_factor = 0.95
for spot_mult_up in [1.10, 1.20, 1.40]:
    spot_mult_down = calcBalancedDownStep(spot_mult_up, discount_factor)
    p_up = (1 / discount_factor - spot_mult_down) / (spot_mult_up - spot_mult_down)
    mean = p_up * spot_mult_up + (1 - p_up) * spot_mult_down
    std = np.sqrt(p_up * (spot_mult_up - mean)**2 + (1 - p_up) * (spot_mult_down - mean)**2)
    print(f"u = {spot_mult_up:.2f}  d = {spot_mult_down:.4f}  p_up = {p_up:.2f}  "
          f"mean = {mean:.4f}  std = {std:.4f}  u - 1/DF = {spot_mult_up - 1/discount_factor:.4f}")

# the up step of the lecture (cell 20): sigma = 0.1, 7 steps per year
steps_per_year = 7
dt = 1 / steps_per_year
discount_factor = 0.95 ** dt
spot_mult_up = np.exp((0.05 - 0.5 * 0.1 ** 2) * dt + 0.1 * np.sqrt(dt))
print(f"lecture: u = {spot_mult_up:.6f}  step size u - 1/DF = {spot_mult_up - 1/discount_factor:.6f}"
      f"  sigma_tree = {(spot_mult_up - 1/discount_factor) / np.sqrt(dt):.4f}")

**(C2) and (C3), one step**: flat up to $u^* = \max(K/S_0,\ 2/DF - K/S_0)$, then increasing (Lemma 3 of the report). The plot uses `my_plotter` of Homework 1 (HW 2/1, `plotter.py`).

In [ ]:
from plotter import my_plotter

# (C2), (C3) one-step tree: price against the up step u, for three strikes
spot, maturity, df_annual = 1.0, 1.0, 0.95
u_grid = np.linspace(1 / df_annual + 1e-6, 2 / df_annual - 1e-6, 400)
strikes = [0.9, spot / df_annual, 1.2]                  # in the money, ATM forward, out of the money
prices = [[balanced_tree_call_price(spot, K, maturity, 1, df_annual, u) for u in u_grid]
          for K in strikes]

# the formula of Lemma 3 gives the same prices
for K, p in zip(strikes, prices):
    d_grid = 2 / df_annual - u_grid
    formula = df_annual / 2 * (np.maximum(u_grid * spot - K, 0) + np.maximum(d_grid * spot - K, 0))
    u_star = max(K / spot, 2 / df_annual - K / spot)
    print(f"K = {K:.4f}: max |tree - formula| = {np.max(np.abs(np.array(p) - formula)):.1e},"
          f"  u* = {u_star:.4f},  price range [{p[0]:.4f}, {p[-1]:.4f}]")

fig, ax = my_plotter(u_grid, prices,
                     layout={"title": "One-step balanced tree: call price against the up step",
                             "x_label": "up step $u$", "y_label": "call price", "hline": None,
                             "grid": True, "figsize": (8, 4), "title_fontsize": 13,
                             "label_fontsize": 11, "legend_fontsize": 10, "linewidth": 2},
                     names=["K = 0.9", "K = 1/0.95 (ATM forward)", "K = 1.2"])

**(C2) and (C3), several steps**, and the Black-Scholes price of Homework 1

In [ ]:
# (C2), (C3) N-step trees against the tree volatility, and the Black-Scholes price of Homework 1
spot, strike, maturity, df_annual = 1.0, 1.0, 1.0, 0.95
r = -np.log(df_annual)                                   # the same flat interest rate
sigma_grid = np.linspace(0.005, 0.6, 120)
curves, names = [], []
for steps in [1, 2, 5, 50]:
    curves.append([balanced_tree_call_price(spot, strike, maturity, steps, df_annual,
                                            up_step_from_tree_vol(s, maturity, steps, df_annual))
                   for s in sigma_grid])
    names.append(f"tree, N = {steps}")
curves.append([black_scholes_eur_call(r=r, T=maturity, S0=spot, sigma=s, K=strike)[0]
               for s in sigma_grid])         # one sigma at a time (assert sigma > 0)
names.append("Black-Scholes (Homework 1)")

for s in [0.1, 0.2, 0.3]:
    i = np.argmin(np.abs(sigma_grid - s))
    print(f"sigma = {sigma_grid[i]:.3f}: " + "  ".join(f"{c[i]:.4f}" for c in curves))
print("increasing:", [bool(np.all(np.diff(c) >= -1e-12)) for c in curves])

fig, ax = my_plotter(sigma_grid, curves,
                     layout={"title": "Call price against the tree volatility (S0 = K = 1, T = 1)",
                             "x_label": "tree volatility $(u - 1/DF)/\\sqrt{\\Delta t}$, or $\\sigma$",
                             "y_label": "call price", "hline": None, "grid": True, "figsize": (8, 4),
                             "title_fontsize": 13, "label_fontsize": 11, "legend_fontsize": 10,
                             "linewidth": 2, "linestyles": ["-", "-", "-", "-", "--"],
                             "colors": ["tab:blue", "tab:orange", "tab:green", "tab:red", "black"]},
                     names=names)